# 04 – Statistical Inference (Student Retention)

**Objective**
Apply comparison-of-means, comparison-of-proportions, comparison-of-variances and ANOVA on the original (pre-SMOTE) dataset. For every test: state hypothesis, justify choice, run it, compute effect size, interpret, give a business implication.

**Inputs**
`data/preprocessed_university_student_retention.csv` ← original, NOT SMOTE-balanced

In [2]:
%pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 5.9 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [statsmodels] [statsmodels]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 0 – Imports & Load

In [3]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

df = pd.read_csv("../data/preprocessed_university_student_retention.csv")
TARGET = "next_semester_dropout"
df[TARGET] = df[TARGET].astype(int)
print("Shape:", df.shape, "\nClass balance:\n", df[TARGET].value_counts(normalize=True).mul(100).round(2))

Shape: (2134, 22) 
Class balance:
 next_semester_dropout
0    83.08
1    16.92
Name: proportion, dtype: float64


## Step 1 – Assumption Checks

In [4]:
NUMERIC = ["age", "gpa_current", "gpa_history", "attendance_rate",
           "credits_completed", "enrollment_gap_months",
           "academic_warning_count", "advisor_meeting_count"]

def shapiro_safe(s):
    s = s.dropna()
    if len(s) > 5000: s = s.sample(5000, random_state=42)
    return stats.shapiro(s)

normality = pd.DataFrame({
    "feature": NUMERIC,
    "shapiro_p": [shapiro_safe(df[c]).pvalue for c in NUMERIC],
})
normality["normal_at_0.05"] = normality["shapiro_p"] > 0.05
print(normality)

def levene(col):
    return stats.levene(df.loc[df[TARGET]==0, col].dropna(),
                        df.loc[df[TARGET]==1, col].dropna()).pvalue

variance_eq = pd.DataFrame({
    "feature": NUMERIC,
    "levene_p": [levene(c) for c in NUMERIC],
})
variance_eq["equal_var"] = variance_eq["levene_p"] > 0.05
print(variance_eq)

                  feature     shapiro_p  normal_at_0.05
0                     age  3.051851e-28           False
1             gpa_current  3.672340e-25           False
2             gpa_history  8.215800e-22           False
3         attendance_rate  1.384722e-24           False
4       credits_completed  7.851911e-26           False
5   enrollment_gap_months  2.039593e-47           False
6  academic_warning_count  1.402609e-40           False
7   advisor_meeting_count  2.541446e-34           False
                  feature  levene_p  equal_var
0                     age  0.606415       True
1             gpa_current  0.805364       True
2             gpa_history  0.738474       True
3         attendance_rate  0.636567       True
4       credits_completed  0.613775       True
5   enrollment_gap_months  0.073263       True
6  academic_warning_count  0.770211       True
7   advisor_meeting_count  0.843269       True


## Step 2 – Comparison of Means

In [5]:
def cohen_d(g0, g1):
    n0, n1 = len(g0), len(g1)
    pooled = np.sqrt(((n0-1)*g0.var(ddof=1) + (n1-1)*g1.var(ddof=1)) / (n0+n1-2))
    return (g1.mean() - g0.mean()) / pooled if pooled > 0 else 0

def t_test_report(col):
    g0 = df.loc[df[TARGET]==0, col].dropna()
    g1 = df.loc[df[TARGET]==1, col].dropna()
    t, p = stats.ttest_ind(g0, g1, equal_var=False)
    return {"feature": col, "mean_retain": g0.mean(), "mean_dropout": g1.mean(),
            "t": t, "p": p, "cohens_d": cohen_d(g0, g1)}

def mw_report(col):
    g0 = df.loc[df[TARGET]==0, col].dropna()
    g1 = df.loc[df[TARGET]==1, col].dropna()
    u, p = stats.mannwhitneyu(g0, g1, alternative="two-sided")
    n0, n1 = len(g0), len(g1)
    r = 1 - (2*u)/(n0*n1)
    return {"feature": col, "median_retain": g0.median(),
            "median_dropout": g1.median(), "U": u, "p": p, "rank_biserial_r": r}

# Parametric for ~symmetric features
t_rows = [t_test_report(c) for c in
          ["gpa_current", "gpa_history", "attendance_rate",
           "credits_completed", "age", "advisor_meeting_count"]]
# Non-parametric for skewed features
mw_rows = [mw_report(c) for c in
           ["enrollment_gap_months", "academic_warning_count"]]

means_df = pd.DataFrame(t_rows + mw_rows)
means_df

,feature,mean_retain,mean_dropout,t,p,cohens_d,median_retain,median_dropout,U,rank_biserial_r
0,gpa_current,2.755742,2.763518,-0.186456,0.852160,0.010799,NaN,NaN,NaN,NaN
1,gpa_history,2.505025,2.494598,0.244577,0.806881,-0.014120,NaN,NaN,NaN,NaN
2,attendance_rate,0.751495,0.758975,-0.892974,0.372290,0.052077,NaN,NaN,NaN,NaN
3,credits_completed,70.288212,67.819945,1.477959,0.140026,-0.085111,NaN,NaN,NaN,NaN
4,age,23.411168,23.337950,0.365314,0.715027,-0.021243,NaN,NaN,NaN,NaN
5,advisor_meeting_count,2.460801,2.545706,-0.858562,0.390983,0.049902,NaN,NaN,NaN,NaN
6,enrollment_gap_months,NaN,NaN,NaN,0.118561,NaN,1.0,0.0,335513.0,-0.048391
7,academic_warning_count,NaN,NaN,NaN,0.205927,NaN,2.0,2.0,306959.5,0.040831


## Step 3 – Comparison of Proportions (Chi-Square)

⚠️ If `cochran_safe == False` for any row, switch to Fisher's exact test for 2×2 tables, or merge sparse categories before re-running chi-square.

In [6]:
def chi_report(col):
    ct = pd.crosstab(df[col], df[TARGET])
    chi2, p, dof, exp = stats.chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape)-1))) if min(ct.shape) > 1 else 0
    safe = ((exp < 5).sum() / exp.size) < 0.20
    return {"feature": col, "chi2": chi2, "dof": dof, "p": p,
            "cramers_v": v, "cochran_safe": safe}

chi_df = pd.DataFrame([chi_report(c) for c in
                       ["gender", "financial_aid_status", "major"]])
chi_df

,feature,chi2,dof,p,cramers_v,cochran_safe
0,gender,6.961444,2,0.030785,0.057115,True
1,financial_aid_status,3.480305,1,0.062103,0.040384,True
2,major,1.290113,4,0.863051,0.024588,True


## Step 4 – Comparison of Variances (Levene's)

In [7]:
var_rows = []
for c in ["gpa_current", "attendance_rate", "credits_completed"]:
    g0, g1 = df.loc[df[TARGET]==0, c].dropna(), df.loc[df[TARGET]==1, c].dropna()
    stat, p = stats.levene(g0, g1, center="median")   # Brown-Forsythe, robust
    var_rows.append({"feature": c, "var_retain": g0.var(ddof=1),
                     "var_dropout": g1.var(ddof=1),
                     "levene_stat": stat, "p": p})
var_df = pd.DataFrame(var_rows)
var_df

,feature,var_retain,var_dropout,levene_stat,p
0,gpa_current,0.517680,0.522513,0.060733,0.805364
1,attendance_rate,0.020528,0.021153,0.223324,0.636567
2,credits_completed,842.192846,835.375823,0.254789,0.613775


## Step 5 – ANOVA + Kruskal-Wallis (Multi-Group)

In [8]:
def anova_report(num, group):
    groups = [g[num].dropna().values for _, g in df.groupby(group)]
    f, p = stats.f_oneway(*groups)
    grand_mean = df[num].mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    ss_total = ((df[num] - grand_mean)**2).sum()
    return {"feature": num, "group": group,
            "F": f, "p": p, "eta_squared": ss_between / ss_total}

anova_df = pd.DataFrame([
    anova_report("gpa_current", "dropout_risk"),
    anova_report("attendance_rate", "dropout_risk"),
    anova_report("academic_warning_count", "major"),
])
print(anova_df)

# Post-hoc: Tukey HSD on the first one
tukey = pairwise_tukeyhsd(df["gpa_current"], df["dropout_risk"], alpha=0.05)
print(tukey.summary())

                  feature         group         F         p  eta_squared
0             gpa_current  dropout_risk  0.003236  0.954642     0.000002
1         attendance_rate  dropout_risk  0.193020  0.660459     0.000091
2  academic_warning_count         major  1.046266  0.381816     0.001962
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper reject
--------------------------------------------------
 False   True    0.002 0.9546 -0.0661  0.07  False
--------------------------------------------------


If normality fails for the ANOVA groups → swap to Kruskal-Wallis:

In [9]:
def kw_report(num, group):
    groups = [g[num].dropna().values for _, g in df.groupby(group)]
    h, p = stats.kruskal(*groups)
    return {"feature": num, "group": group, "H": h, "p": p}

## Step 6 – Multiple Testing Correction

In [10]:
all_p = (
    means_df["p"].tolist()
    + chi_df["p"].tolist()
    + var_df["p"].tolist()
    + anova_df["p"].tolist()
)
reject, p_adj, *_ = multipletests(all_p, alpha=0.05, method="fdr_bh")
print(f"Significant after FDR: {sum(reject)} / {len(reject)}")

Significant after FDR: 0 / 17


## Step 7 – Master Summary Table

In [11]:
rows = []
for _, r in means_df.iterrows():
    rows.append({"family": "Means", "feature": r["feature"],
                 "statistic": f"t={r['t']:.3f}" if "t" in r else f"U={r['U']:.1f}",
                 "p": r["p"], "effect": f"d={r['cohens_d']:.2f}" if "cohens_d" in r else f"r={r['rank_biserial_r']:.2f}"})
for _, r in chi_df.iterrows():
    rows.append({"family": "Proportions", "feature": r["feature"],
                 "statistic": f"χ²={r['chi2']:.2f}, df={r['dof']}",
                 "p": r["p"], "effect": f"V={r['cramers_v']:.2f}"})
for _, r in var_df.iterrows():
    rows.append({"family": "Variances", "feature": r["feature"],
                 "statistic": f"L={r['levene_stat']:.2f}",
                 "p": r["p"], "effect": "(see variances)"})
for _, r in anova_df.iterrows():
    rows.append({"family": "ANOVA", "feature": r["feature"],
                 "statistic": f"F={r['F']:.2f}",
                 "p": r["p"], "effect": f"η²={r['eta_squared']:.2f}"})

summary = pd.DataFrame(rows)
summary.to_csv("task4_inference_summary.csv", index=False)
summary

,family,feature,statistic,p,effect
0,Means,gpa_current,t=-0.186,0.852160,d=0.01
1,Means,gpa_history,t=0.245,0.806881,d=-0.01
2,Means,attendance_rate,t=-0.893,0.372290,d=0.05
3,Means,credits_completed,t=1.478,0.140026,d=-0.09
4,Means,age,t=0.365,0.715027,d=-0.02
5,Means,advisor_meeting_count,t=-0.859,0.390983,d=0.05
6,Means,enrollment_gap_months,t=nan,0.118561,d=nan
7,Means,academic_warning_count,t=nan,0.205927,d=nan
8,Proportions,gender,"χ²=6.96, df=2",0.030785,V=0.06
9,Proportions,financial_aid_status,"χ²=3.48, df=1",0.062103,V=0.04


## Step 8 – Report-Writing Prompt

For every row in summary, write one paragraph covering:

1. Test used and which family it belongs to (means / proportions / variances / ANOVA).
2. H₀ and H₁ in plain English.
3. Result (statistic, p-value, effect size).
4. Interpretation in words.
5. Business implication — what should the university do?

If p > 0.05 after FDR correction → explicitly say "no statistically detectable difference; the data do not support using this feature for retention decisions."

**Done When…**
* Every test family from the brief is represented (means, proportions, variances, ANOVA)
* Hypothesis stated for every test
* Effect size reported alongside every p-value
* Multiple testing correction applied
* Master table saved
* One paragraph per test written for the report
* Business implication included for every significant finding